In [38]:
from dotenv import load_dotenv
import os
from euriai.langchain import create_chat_model
import time
from euriai.langchain import EuriaiEmbeddings

app_dir = os.path.join(os.getcwd(), "app")
load_dotenv(os.path.join(app_dir, ".env"))

api_key = os.getenv("key") 

chat_model = create_chat_model(api_key=api_key, model="gpt-4.1-nano", temperature=0.7)
model = chat_model

embeddings = EuriaiEmbeddings(
    api_key=api_key,
    model="text-embedding-3-small"
)

In [39]:
from langchain_classic import hub

prompt = hub.pull("hwchase17/openai-tools-agent")
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [40]:

# Inspect the prompt in detail
print("=== Input Variables ===")
print(prompt.input_variables)

print("\n=== Messages ===")
for i, m in enumerate(prompt.messages):
    print(f"\n[{i}] Type: {type(m).__name__}")
    if hasattr(m, 'prompt'):
        print(f"    Template : {m.prompt.template}")
    elif hasattr(m, 'variable_name'):
        print(f"    Placeholder: {{{m.variable_name}}}")
    elif hasattr(m, 'content'):
        print(f"    Content: {m.content}")


=== Input Variables ===
['agent_scratchpad', 'input']

=== Messages ===

[0] Type: SystemMessagePromptTemplate
    Template : You are a helpful assistant

[1] Type: MessagesPlaceholder
    Placeholder: {chat_history}

[2] Type: HumanMessagePromptTemplate
    Template : {input}

[3] Type: MessagesPlaceholder
    Placeholder: {agent_scratchpad}


In [41]:
## `agent_scratchpad` in a normal LLM call
## `agent_scratchpad` is just a plain list of messages. You can pass the same messages directly to any LLM — no agent framework needed.

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

llm = model

# Manually build the full message list — this is exactly what agent_scratchpad does
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="What dishes do they serve?"),

    # --- this part IS the agent_scratchpad ---
    AIMessage(content="", additional_kwargs={
        "tool_calls": [{
            "id": "call_abc123",
            "type": "function",
            "function": {
                "name": "ragagent",
                "arguments": '{"query": "dishes served at the restaurant"}'
            }
        }]
    }),
    ToolMessage(
        tool_call_id="call_abc123",
        content="The restaurant serves pasta, pizza, and tiramisu."
    ),
    # -----------------------------------------
]

# Ask the LLM to produce a final answer given the tool result
response = llm.invoke(messages)
print(response.content)


The restaurant serves pasta, pizza, and tiramisu.


## Understanding the `agent_scratchpad` messages

The scratchpad is always a **paired exchange** between the LLM and a tool:

---

### Message 1 — `AIMessage` with `tool_calls`
This is what the **LLM itself generates** when it decides it needs to call a tool instead of answering directly.

```python
AIMessage(content="", additional_kwargs={
    "tool_calls": [{
        "id": "call_abc123",       # unique ID linking this call to its result
        "type": "function",
        "function": {
            "name": "ragagent",    # which tool to call
            "arguments": '{"query": "dishes served at the restaurant"}'  # what to pass
        }
    }]
})
```
- `content=""` → the LLM has nothing to say yet, it's delegating to a tool
- `tool_calls` → the LLM's structured request saying *"run this tool with these args"*
- `id` → a unique tag so the result can be matched back to this request

---

### Message 2 — `ToolMessage`
This is the **tool's response**, linked back to the AIMessage via the same `tool_call_id`.

```python
ToolMessage(
    tool_call_id="call_abc123",   # matches the id above
    content="The restaurant serves pasta, pizza, and tiramisu."  # what the tool returned
)
```

---

### Where do these come from in real life?

In an actual agent run, **you never write these manually**. The framework generates them automatically:

```
LLM output → has tool_calls?
                    │ YES
                    ▼
        Framework calls the tool
                    │
                    ▼
        ToolMessage created with result
                    │
                    ▼
        Both messages appended to agent_scratchpad
                    │
                    ▼
        LLM called again with full context → final answer
```

The manual example in the code above just **simulates** what the framework would produce, so you can understand what gets passed to the LLM.

## What is `agent_scratchpad`?

`agent_scratchpad` is the **working memory of the agent within a single run**. It holds the intermediate steps — tool calls and their results — that the agent has taken before arriving at a final answer.

### How it works:
```
User asks a question
    │
    ▼
[1] Agent decides to call a tool
    → AIMessage with tool_calls: [{"name": "ragagent", "arguments": ...}]
    │
    ▼
[2] Tool runs and returns a result
    → ToolMessage: "The restaurant serves pasta, pizza, tiramisu."
    │
    ▼
[3] Both messages go into agent_scratchpad
    and are sent back to the LLM to formulate a final answer
```

### Key differences:
| Placeholder | Purpose | Scope |
|---|---|---|
| `chat_history` | Memory **across** turns (previous Q&A) | Persists between requests |
| `agent_scratchpad` | Memory **within** a single turn (tool calls so far) | Reset on every new question |

> When using LangGraph or `AgentExecutor`, `agent_scratchpad` is **filled automatically** — you never populate it yourself.

In [42]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# Example: invoke the prompt with chat_history and agent_scratchpad populated
formatted = prompt.invoke({
    "input": "What dishes do they serve?",

    # Simulate a prior turn in the conversation
    "chat_history": [
        HumanMessage(content="Who is the owner of the restaurant?"),
        AIMessage(content="The owner of the restaurant is John Smith."),
    ],

    # Simulate an agent that already made one tool call and got a result
    "agent_scratchpad": [
        AIMessage(content="", additional_kwargs={
            "tool_calls": [{
                "id": "call_abc123",
                "type": "function",
                "function": {
                    "name": "ragagent",
                    "arguments": '{"query": "dishes served at the restaurant"}'
                }
            }]
        }),
        ToolMessage(
            tool_call_id="call_abc123",
            content="The restaurant serves pasta, pizza, and tiramisu."
        ),
    ]
})

# Show the resulting messages that would be sent to the LLM
for msg in formatted.messages:
    print(f"[{type(msg).__name__}]: {msg.content}")


[SystemMessage]: You are a helpful assistant
[HumanMessage]: Who is the owner of the restaurant?
[AIMessage]: The owner of the restaurant is John Smith.
[HumanMessage]: What dishes do they serve?
[AIMessage]: 
[ToolMessage]: The restaurant serves pasta, pizza, and tiramisu.


In [43]:
print(prompt)

input_variables=['agent_scratchpad', 'input'] optional_variables=['chat_history'] input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')] | typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')] | typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')] | typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')] | t

In [44]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders.directory import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = DirectoryLoader("./data", glob="**/*.txt")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=120,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)
chunks = text_splitter.split_documents(docs)

embedding_function = embeddings
model = model

db = Chroma.from_documents(chunks, embedding_function)
retriever = db.as_retriever()

In [45]:
# from langchain_core.tools.retriever import create_retriever_tool

# tool = create_retriever_tool(
#     retriever=retriever, name="ragagent", description="performs RAG on a small dataset"
# )

from langchain.tools import tool

@tool
def retrieve_query(query: str) -> str:
    """Search and return information about a query."""
    docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

retriever_tool = retrieve_query
    
tools = [retriever_tool]

In [46]:
# Use the euriai model explicitly — avoids picking up a stale ChatOpenAI from the kernel
llm = chat_model
print(type(llm))  # should show EuriaiChatModel


<class 'euriai.langchain.EuriaiChatModel'>


## Creating an Agent in LangChain

### ✅ Modern way — LangGraph `create_react_agent` (recommended)
```python
from langgraph.prebuilt import create_react_agent

# Simple usage — no prompt needed
agent = create_react_agent(model=llm, tools=tools)
agent.invoke({"messages": [("human", "your question")]})

# With a custom system prompt
agent = create_react_agent(model=llm, tools=tools, prompt="You are a helpful assistant.")
```

### ❌ Deprecated — `create_tool_calling_agent` + `AgentExecutor`
```python
from langchain.agents import AgentExecutor, create_tool_calling_agent

agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools)
agent_executor.invoke({"input": "your question"})  # ❌ AgentExecutor is deprecated
```

### ❌ Older deprecated — `create_openai_tools_agent`
```python
from langchain.agents import AgentExecutor, create_openai_tools_agent

agent = create_openai_tools_agent(llm, tools, prompt)  # ❌ deprecated
agent_executor = AgentExecutor(agent=agent, tools=tools)  # ❌ deprecated
```

| Approach | Status | Custom Prompt |
|---|---|---|
| LangGraph `create_react_agent` | ✅ Current | ✅ via `prompt=` parameter |
| `create_tool_calling_agent` + `AgentExecutor` | ❌ Deprecated | ✅ Yes |
| `create_openai_tools_agent` + `AgentExecutor` | ❌ Deprecated | ✅ Yes |

In [47]:
# ✅ Classic ReAct PromptTemplate → use langchain.agents.create_react_agent + AgentExecutor
# This prompt has {tools}, {tool_names}, {input}, {agent_scratchpad}
# AgentExecutor fills {tools}, {tool_names}, {agent_scratchpad} automatically
# You only need to pass "input"
from langchain.agents import create_react_agent, AgentExecutor

agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# ✅ Only pass "input" — everything else is handled automatically
result = agent_executor.invoke({"input": "Who is the owner of the restaurant?"})
print(result["output"])
# ✅ Use "messages" key, not "input"
result = agent.invoke({})
print(result["messages"][-1].content)

print(result)
print(prompt.input_variables)  # should show: ['agent_scratchpad', 'input', 'tool_names', 'tools']

C:\Users\C90008809\AppData\Local\Temp\ipykernel_28972\2146506664.py:7: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


C:\Users\C90008809\AppData\Local\Temp\ipykernel_28972\2146506664.py:7: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


I'm sorry, but I don't have information about the owner of the restaurant. Could you please specify the name of the restaurant?
{'messages': [HumanMessage(content='Who is the owner of the restaurant?', additional_kwargs={}, response_metadata={}, id='5faf45ab-27f9-4d83-ba11-9c856cf96b4c'), AIMessage(content="I'm sorry, but I don't have information about the owner of the restaurant. Could you please specify the name of the restaurant?", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 31, 'completion_tokens': 25, 'total_tokens': 56}, 'model_name': 'gpt-4.1-nano', 'system_fingerprint': None, 'finish_reason': 'stop', 'model': 'gpt-4.1-nano', 'created': 1773317188}, id='lc_run--019ce1f0-e95f-7550-87a2-ecc854be6b3f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 31, 'output_tokens': 25, 'total_tokens': 56})]}


In [ ]:
# ### Deprecated

# from langchain.agents import AgentExecutor, create_openai_tools_agent

# agent = create_openai_tools_agent(llm, tools, prompt)
# agent_executor = AgentExecutor(agent=agent, tools=tools)

In [ ]:
# # ### Deprecated
# agent_executor.invoke({"input": "Who is the owner of the restaurant?"})